
The entire set of scripts (00–03, four scripts in total) processes three main inputs: the original survey data, the reweighted survey trips, and a TAZ-to-district lookup table. The primary goal is to create additional columns that support summarizing and analyzing the survey data. These include trip purpose in Production-Attraction (PA) format, mode of access in PA format, and rider information. The output is a more detailed and comprehensive dataset suitable for further analysis. See `output/Final_2023_OnBoardSurvey_Dataset.csv`.

In addition, the scripts generate high-level summary tables that support the final report. Look for sections labeled `Table xx in Report` to see how each output corresponds to the Mid-Coast After Study Report.


### Structure


- **High-Level Summary**
  - Project Trips Summary  
    - *Table 3 in Report*

- **Station Boardings and Alightings**
  - *Tables 7, 8, and 21 in Report*

- **District-to-District Summary**
  - *Table 6 and Appendix A in Report*

- **Station Portfolio Analysis** *(Section 2.1.2.1 in Report)*
  - Generates final full datasets
  - Reference: `4_2_Rider_Portfolio_by_Station_Archive.xlsx` for the Report


# High Level Summary

**Purpose:** Summarize trips using data from the 2024 SANDAG survey and reweighted records, categorized by trip purpose, auto ownership, and mode of access.

**I/O Files:**

Input Files:

1. `od_20240422_sandagca_weighted_combined_draftfinal.xlsx` - Raw survey file.
2. `2023OBS_ReWeighted_2024-05-24.xlsx` - This includes the reweighted trips for each record.
3. `2023OBS_wTAZ_Districts.csv` - This includes TAZ and District lookup for each record.
4. `2023OBS_project_trip_dataset.csv` - This includes updated Mid-Coast project defination.

Intermediate/Helper Files:

- `Interim_2023_OnBoardSurvey_Dataset_x.csv` : This file includes all survey records, not only the raw survey fields but also the processed columns. It is an intermediate file (an output of this script) that is used as input in the next script for the data model.

Output Files:

- `Table_x.csv`: Summary table used in the report. The table index should match the one in the *Mid-Coast After Survey Report*.

In [1]:
import warnings
warnings.simplefilter("always")

### Read Inputs

In [2]:
## Data Preparation and Summary
import pandas as pd
import numpy as np

# Set directory
data_dir = "../input/"
interim_dir = '../output/interim/'
output_dir = '../output/'

df_full = pd.read_excel(data_dir + "od_20240422_sandagca_weighted_combined_draftfinal.xlsx", sheet_name="OD_RESULTS", engine='openpyxl')

In [3]:
df_project = pd.read_csv(output_dir + '2023OBS_project_trip_dataset.csv')

cols = [
    'ID',
    'xfer_position',
    'xfer_on_name',
    'xfer_off_name',
    'xfer_on_midcoast',
    'xfer_direction',
    'AGENCY',
    'ROUTE_NUMBER',
    'DIRECTION',
    'project_trip_update',
    'stop_on_project',
    'stop_off_project',
    'direction_od_project',
    'origin_transport_mode',
    'destin_transport_mode',
    'last_position',
    'access_w_xfer_od_project',
    'egress_w_xfer_od_project',
]

df = df_full.merge(df_project[cols],on='ID',how='left')
df.shape

(35260, 218)

In [4]:
df_weight = pd.read_excel(data_dir + "2023OBS_ReWeighted_2024-05-24.xlsx",
                          sheet_name="Sheet1", engine='openpyxl')

In [5]:
df = pd.merge(df, df_weight[['ID', 'REWEIGHTED_LINKED', 'REWEIGHTED_UNLINKED']],
              on='ID', how='left')

In [6]:
df[['AGENCY', 'AGENCY_CODE','ROUTE_NUMBER', 'DIRECTION']] = df['ROUTE_DIRECTION[Code]'].str.split('_', expand=True)

In [7]:
print(df['REWEIGHTED_LINKED'].sum())
print(df['REWEIGHTED_UNLINKED'].sum())
print(df[df['project_trip_update']=='yes']['REWEIGHTED_LINKED'].sum())
print(df[df['project_trip_update']=='yes']['REWEIGHTED_UNLINKED'].sum())

225994.952324302
280605.804192621
24735.73668767653
30474.896296270348


### Define Trip Purpose

In [8]:
# Creating trip purpose: HBW, HBO, and NHB
df['TRIP_PURPOSE'] = np.where(((df['ORIGIN_PLACE_TYPE'] == 'Your HOME') & (df['DESTIN_PLACE_TYPE'] == 'Your usual WORKPLACE')) |
                              ((df['ORIGIN_PLACE_TYPE'] == 'Your usual WORKPLACE') & (df['DESTIN_PLACE_TYPE'] == 'Your HOME')), 'HBW',
                              
                     np.where(((df['ORIGIN_PLACE_TYPE'] == 'Your HOME') & (df['DESTIN_PLACE_TYPE'] == 'School (K-12) (students only)')) |
                              ((df['ORIGIN_PLACE_TYPE'] == 'School (K-12) (students only)') & (df['DESTIN_PLACE_TYPE'] == 'Your HOME')), 'HBSC',                              
                              
                     np.where(((df['ORIGIN_PLACE_TYPE'] == 'Your HOME') & (df['DESTIN_PLACE_TYPE'] == 'College / University (students only)')) |
                              ((df['ORIGIN_PLACE_TYPE'] == 'College / University (students only)') & (df['DESTIN_PLACE_TYPE'] == 'Your HOME')), 'HBU',                               

                     np.where(((df['ORIGIN_PLACE_TYPE'] == 'Your HOME') & (df['DESTIN_PLACE_TYPE'] == 'Medical Services / Hospital (non-work)')) |
                              ((df['ORIGIN_PLACE_TYPE'] == 'Medical Services / Hospital (non-work)') & (df['DESTIN_PLACE_TYPE'] == 'Your HOME')), 'HBM', 
                                       
                     np.where(
                             (
                                (df['ORIGIN_PLACE_TYPE'] == 'Your HOME') & 
                                (
                                    (df['DESTIN_PLACE_TYPE'] != 'Your usual WORKPLACE') &
                                    (df['DESTIN_PLACE_TYPE'] != 'School (K-12) (students only)') &
                                    (df['DESTIN_PLACE_TYPE'] != 'College / University (students only)') &
                                    (df['DESTIN_PLACE_TYPE'] != 'Medical Services / Hospital (non-work)')
                                )
                             ) |
                             (
                                (
                                    (df['ORIGIN_PLACE_TYPE'] != 'Your usual WORKPLACE') &
                                    (df['ORIGIN_PLACE_TYPE'] != 'School (K-12) (students only)') &
                                    (df['ORIGIN_PLACE_TYPE'] != 'College / University (students only)') &
                                    (df['ORIGIN_PLACE_TYPE'] != 'Medical Services / Hospital (non-work)')
                                ) &
                                (df['DESTIN_PLACE_TYPE'] == 'Your HOME')
                             ),
                         'HBO',
                         'NHB')))))

In [9]:
df['TRIP_PURPOSE_AGG'] = np.where(df['TRIP_PURPOSE'].isin(['HBU', 'HBSC', 'HBM']), 'HBO', df['TRIP_PURPOSE'])

In [10]:
# Creating production and attraction flag
# If the trip starts from Production-to-Attraction (PA), then flag=1. If it is Attraction-to-Production (AP), then flag=0.
# Code: If the trip is home-based and the destination is home, the trip is AP format, otherwise it is PA.
df['PA_FLAG'] = np.where(df['TRIP_PURPOSE'].isin(['HBW', 'HBO', 'HBU', 'HBSC', 'HBM']) &
                         df['DESTIN_PLACE_TYPE'].isin(['Your HOME']), 0, 1)

# print(df['PA_FLAG'].value_counts(dropna=False))

### Define Mode of Access in Production-Attraction format

In [11]:
# Creating access mode: Walk, XFER, KNR, and PNR
df['ACCESS_MODE_OD'] = np.where(df['ORIGIN_TRANSPORT'] == 'Walk', 'Walk',
                    
                       np.where(df['ORIGIN_TRANSPORT'].isin(['Was dropped off by someone',
                                                             'Uber, Lyft, etc. (pool or shared)',
                                                             'Taxi',
                                                             'Other shuttle',
                                                             'Uber, Lyft, etc. (private)',
                                                             'Electric vehicle shuttle']), 'KNR',
                        
                       np.where(df['ORIGIN_TRANSPORT'].isin(['Drove alone and parked',
                                                             'Drove or rode with others and parked']), 'PNR', 'Walk')))

In [12]:
df['EGRESS_MODE_OD'] = np.where(df['DESTIN_TRANSPORT'] == 'Walk', 'Walk',
                    
                       np.where(df['DESTIN_TRANSPORT'].isin(['Be picked up by someone',
                                                             'Uber, Lyft, etc. (pool or shared)',
                                                             'Taxi',
                                                             'Other shuttle',
                                                             'Uber, Lyft, etc. (private)',
                                                             'Electric vehicle shuttle']), 'KNR',
                             
                       np.where(df['DESTIN_TRANSPORT'].isin(['Get in a parked vehicle & drive alone',
                                                             'Get in a parked vehicle & drive/ride w/others']), 'PNR', 'Walk')))                        

In [13]:
# Adjusting access mode for AP trips
df['ACCESS_MODE_PA'] = np.where(df['PA_FLAG'] == 0, df['EGRESS_MODE_OD'], df['ACCESS_MODE_OD'])

# Adjusting egress mode for AP trips
df['EGRESS_MODE_PA'] = np.where(df['PA_FLAG'] == 0, df['ACCESS_MODE_OD'], df['EGRESS_MODE_OD'])

### Other project related columns

In [14]:
# Adjusting access mode for AP trips
df['access_w_xfer_pa_project'] = np.where(df['PA_FLAG'] == 0, df['egress_w_xfer_od_project'], df['access_w_xfer_od_project'])

# Adjusting egress mode for AP trips
df['egress_w_xfer_pa_project'] = np.where(df['PA_FLAG'] == 0, df['access_w_xfer_od_project'], df['egress_w_xfer_od_project'])

In [15]:
df['stop_prod_project'] = np.where(df['PA_FLAG'] == 0, df['stop_off_project'], df['stop_on_project'])
df['stop_attr_project'] = np.where(df['PA_FLAG'] == 0, df['stop_on_project'], df['stop_off_project'])

In [16]:
# Creating the survey route direction in PA format
df['direction_od_reverse'] = None
df['direction_pa_project'] = None
df['direction_od_reverse'] = np.where(df['direction_od_project']=='NB', 'SB', np.where(df['direction_od_project']=='SB', 'NB', None))
df['direction_pa_project'] = np.where(df['PA_FLAG'] == 0, df['direction_od_reverse'], df['direction_od_project'])

In [17]:
df = df.drop(columns=['direction_od_reverse'])

### Define Auto Ownership

In [18]:
# Creating auto ownership: 0-car, 1-car, 2-car and plus
df['AUTO_OWNERSHIP'] = np.where(df['COUNT_VH_HH'] == 'None (0)', 'None (0)',
                       np.where(df['COUNT_VH_HH'] == 'One (1)', 'One (1)', 'Two or more (2+)'))
                    

## Project Trips Summary

In [19]:
project_trips = df[df['project_trip_update']=='yes'].copy()

by trip purpose

In [20]:
pj_trips_bypurp = project_trips.groupby('TRIP_PURPOSE_AGG').agg({'REWEIGHTED_LINKED': 'sum'}).round(1)
pj_trips_bypurp

,REWEIGHTED_LINKED
TRIP_PURPOSE_AGG,
HBO,12125.7
HBW,8390.5
NHB,4219.6


by auto ownership

In [21]:
pj_trips_byauto = project_trips.groupby('AUTO_OWNERSHIP').agg({'REWEIGHTED_LINKED': 'sum'}).round(1)
pj_trips_byauto

,REWEIGHTED_LINKED
AUTO_OWNERSHIP,
None (0),8612.8
One (1),7265.5
Two or more (2+),8857.4


by mode of access

In [22]:
pj_trips_byaccess = project_trips.groupby('ACCESS_MODE_PA').agg({'REWEIGHTED_LINKED': 'sum'}).round(1)
pj_trips_byaccess

,REWEIGHTED_LINKED
ACCESS_MODE_PA,
KNR,2775.6
PNR,3408.5
Walk,18551.6


### Table 3 in Report 

Table 3 Mid-Coast Project Trips, 2023 Transit On-Board Survey

In [23]:
pivot = pd.pivot_table(
    project_trips,
    index=['AUTO_OWNERSHIP', 'ACCESS_MODE_PA'],
    columns='TRIP_PURPOSE_AGG',
    values='REWEIGHTED_LINKED',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)

# Reset index for better readability
pivot = pivot[['HBW', 'HBO', 'NHB', 'Total']]
# pivot.reset_index().round(1).to_csv(output_dir + 'Table_3_Project_Trips_High_Level_Summary.csv',index=False)
pivot.round(1)

TRIP_PURPOSE_AGG                    HBW      HBO     NHB    Total
AUTO_OWNERSHIP   ACCESS_MODE_PA                                  
None (0)         KNR              103.5    403.5   371.4    878.4
                 PNR               80.2     12.3     0.0     92.5
                 Walk            1973.2   3974.9  1693.8   7641.9
One (1)          KNR              612.1    253.2     0.9    866.1
                 PNR              652.0    530.8     0.5   1183.2
                 Walk            1933.1   2357.3   925.7   5216.2
Two or more (2+) KNR              297.7    705.4    28.0   1031.1
                 PNR              923.8   1209.0     0.0   2132.8
                 Walk            1814.8   2679.4  1199.4   5693.5
Total                            8390.5  12125.7  4219.6  24735.7

In [24]:
df.to_csv(interim_dir + 'Interim_2023_OnBoardSurvey_Dataset_1.csv',index=False)